# Leische — Context-Aware Sarcasm Detection in Code-Switching Social Media Posts

**Single-notebook model pipeline.** Data comes from the companion **uyam**
repository (collection + LLM-ensemble annotation); this notebook auto-detects
it (§3 below). The methodology reference is [docs/MODEL_PLAN.md](docs/MODEL_PLAN.md).

> **STATUS: SMOKE PHASE — no real training, no reportable metrics.**
> The current export (`dataset-v1`) is a 100-row pilot (8 sarcastic, no human
> gold subset). The §10 data-readiness gate FAILS and is enforced in code:
> every number below is a harness/architecture check marked `SMOKE`.

**Contents**
1. Setup & environment
2. Configuration (all switches; thesis baseline defaults, upgrades off)
3. Data — auto-detect uyam, load, validate, readiness gate
4. EDA (rerun on every dataset version)
5. Frozen folds (StratifiedGroupKFold, thread-grouped)
6. Context channels — conversational / temporal / retrieval (+ leakage checks)
7. Model — the 5-stage architecture behind ablation flags
8. Training harness
9. Smoke test 1 — overfit 16 rows (baseline + full model)
10. Smoke test 2 — tiny-settings 5-fold dry run (baseline)
11. Full context model — gates logged, checkpoint round-trip
12. Ablation matrix (8 conditions × 5 folds) + significance tests
13. RQ3 — two-stage sentiment evaluation
14. Verdict & next steps

## 1 · Setup & environment

In [ ]:
import copy
import json
import math
import platform
import random
import subprocess
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer

print(f"python        {sys.version.split()[0]} on {platform.system()} {platform.release()}")
for mod in (torch, transformers, sklearn, pd, np, matplotlib):
    print(f"{mod.__name__:<13} {mod.__version__}")

assert torch.cuda.is_available(), "CUDA GPU required for the smoke training runs"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB | cuda {torch.version.cuda}")
DEVICE = torch.device("cuda")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# determinism check: same seed → bit-identical forward pass
def _seeded_forward():
    set_seed(13)
    return nn.Linear(64, 8)(torch.randn(4, 64))

assert torch.equal(_seeded_forward(), _seeded_forward())
print("determinism check PASSED")

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "run this notebook from the repo root"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

## 2 · Configuration

One dataclass, every switch from the plan. Defaults are the thesis-committed
baseline (MODEL_PLAN §4); every §9 upgrade is a flag that defaults to OFF so
each manuscript claim stays reproducible with upgrades disabled.

In [ ]:
@dataclass
class Config:
    dataset_version: str = "v2"
    run_name: str = "dev"

    # stage 1 — shared encoder (§4.1)
    encoder_name: str = "xlm-roberta-base"
    pooling: str = "mean"            # "mean" | "cls" — both verified in §10
    d_model: int = 256
    max_len_target: int = 192
    max_len_context: int = 96
    max_len_selftext: int = 128

    # RQ2 ablation flags (§7.3)
    use_conv: bool = False
    use_temp: bool = False
    use_ret: bool = False

    # stage 2 — conversational + temporal (§4.2)
    conv_role_embeddings: bool = True
    temporal_k: int = 10
    temporal_lambda_init: float = 0.1
    temporal_lambda_learnable: bool = False  # thesis: fixed decay; learnable = flagged generalization
    missing_channel: str = "zeros"           # "zeros" | "learned"

    # stage 3 — retrieval (§4.3)
    retrieval_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    retrieval_k: int = 5
    retrieval_prototypes: bool = False

    # stage 5 — classifier (§4.5)
    mlp_hidden: int = 256
    dropout: float = 0.2

    # training defaults (§4)
    lr_encoder: float = 2e-5
    lr_heads: float = 1e-4
    warmup_ratio: float = 0.1
    max_epochs: int = 10
    patience: int = 3
    batch_size: int = 8
    grad_accum: int = 2
    fp16: bool = True
    grad_clip: float = 1.0
    seeds: list = field(default_factory=lambda: [13, 42, 7])
    class_weighting: str = "inverse_freq"    # per training fold, never global
    max_steps_per_epoch: int | None = None   # smoke throttle

    # §9 upgrades — ALL OFF by default
    sample_weighting: bool = False
    sample_weights: dict = field(default_factory=lambda: {
        # unanimous_partial = only 2 of 3 annotator models returned a label
        "unanimous": 1.0, "unanimous_partial": 0.85, "majority": 0.9,
        "adjudicator": 0.7, "human": 1.0})
    soft_labels: bool = False
    focal_loss: bool = False
    focal_gamma: float = 2.0
    weighted_sampler: bool = False
    target_pos_frac: float = 0.25
    aux_cue_heads: bool = False
    aux_polarity_shift: bool = False
    aux_language_head: bool = False
    aux_loss_weight: float = 0.25
    freeze_bottom_layers: int = 0
    layerwise_lr_decay: float | None = None
    temperature_scaling: bool = False
    tune_threshold_on_val: bool = False

    # evaluation (§7)
    n_folds: int = 5
    natural_only_metrics: bool = True

    # smoke: True → every metric/artifact prefixed SMOKE; run_cv refuses
    # smoke=False while the §10 gate fails
    smoke: bool = True

    def tag(self) -> str:
        return "SMOKE " if self.smoke else ""

    def to_dict(self) -> dict:
        return asdict(self)


def smoke_cfg(**over) -> Config:
    """Tiny-settings profile: proves the harness, never produces results."""
    base = dict(run_name="smoke", smoke=True, max_len_target=96, max_len_context=64,
                max_len_selftext=64, batch_size=8, grad_accum=1, max_epochs=2,
                patience=1, max_steps_per_epoch=8, seeds=[13])
    base.update(over)
    return Config(**base)


def ablation_cfg(**over) -> Config:
    """Ultra-tiny profile for the 8×5 matrix dry run."""
    base = dict(run_name="ablation", smoke=True, max_len_target=64, max_len_context=48,
                max_len_selftext=48, batch_size=8, grad_accum=1, max_epochs=1,
                patience=1, max_steps_per_epoch=6, seeds=[13])
    base.update(over)
    return Config(**base)


CFG = Config()  # thesis baseline defaults
print(f"config ready — thesis defaults, smoke={CFG.smoke}")

## 3 · Data — load, validate, readiness gate

uyam ships two flat CSVs (`annotated-review.csv`, `uyam_export.csv`).
`tools/build_uyam_export.py` derives the dataset contract from them:

```bash
uv run python tools/build_uyam_export.py      # → data/dataset-v2.jsonl, corpus-v2.jsonl, dataset_card.json
```

Load order: `./data/dataset-vN.jsonl` first, else the sibling
`../uyam/data/annotated/`. Three fields the contract asks for are **not
collected by uyam** and arrive as `null` — `labels.cues`, `aux.tx_sentiment`,
`aux.lid`, `human_gold`; the validator tolerates them and the features that
depend on them stay off (see `docs/UYAM_HANDOFF.md`).

> **Conversational context is currently a corpus rebuild, not the annotator
> snapshot** (`context.source == "corpus_rebuild"`). MODEL_PLAN §11.7 wants the
> embedded snapshot; uyam does not export one yet (handoff H2). The rebuild is
> flagged below and swaps out without touching anything downstream.

In [ ]:
LANGUAGES = ("english", "tagalog", "taglish")
SENTIMENTS = ("positive", "neutral", "negative")
RESOLVED_BY = ("unanimous", "unanimous_partial", "majority", "adjudicator", "human")
CUE_KEYS = ("polarity_inversion", "rhetorical_intent", "contextual_incongruity", "hyperbole")

# Fields uyam does not collect. They are null in every row, so the contract
# treats them as optional and the code paths that consume them stay disabled
# rather than fabricating values (docs/UYAM_HANDOFF.md H1, H2, H5, H6).
UNCOLLECTED = ("labels.cues", "aux.tx_sentiment", "aux.lid", "human_gold")


def locate_data(version: str) -> Path:
    local = ROOT / "data"
    if (local / f"dataset-{version}.jsonl").exists():
        return local
    uyam = ROOT.parent / "uyam" / "data" / "annotated"
    if (uyam / f"dataset-{version}.jsonl").exists():
        print(f"using the sibling uyam export: {uyam}")
        return uyam
    raise FileNotFoundError(
        f"dataset-{version}.jsonl not found in {local} or {uyam} — download the "
        "uyam export into ./data/ or clone uyam next to this repo")


def _read_jsonl(path: Path) -> list[dict]:
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def validate_rows(rows: list[dict]) -> list[str]:
    """Dataset-contract checks; returns a list of problem strings."""
    problems, seen = [], set()
    for i, r in enumerate(rows):
        w = f"row {i} ({r.get('reddit_fullname', '?')})"
        bad = lambda m: problems.append(f"{w}: {m}")
        for key in ("reddit_fullname", "record_type", "submission_fullname", "created_utc",
                    "author_hash", "text", "labels", "reliability", "aux", "context"):
            if key not in r:
                bad(f"missing {key}")
        rid = r.get("reddit_fullname")
        if rid in seen:
            bad("duplicate reddit_fullname")
        seen.add(rid)
        if r.get("sampling_strategy") not in ("natural", "keyword_oversampled", None):
            bad(f"sampling_strategy={r.get('sampling_strategy')!r}")
        if not str(r.get("text", "")).strip():
            bad("empty text")
        lab = r.get("labels") or {}
        if not isinstance(lab.get("sarcastic"), bool):
            bad("labels.sarcastic not bool")
        if lab.get("language") not in LANGUAGES:
            bad(f"labels.language={lab.get('language')!r}")
        for k in ("literal_sentiment", "intended_sentiment"):
            # null = the sentiment vote never resolved; RQ3 filters these out
            # rather than dropping the row from the sarcasm task (§13)
            if lab.get(k) is not None and lab.get(k) not in SENTIMENTS:
                bad(f"labels.{k}={lab.get(k)!r}")
        if lab.get("cues") is not None:  # optional — uyam does not collect them
            for k in CUE_KEYS:
                if not isinstance(lab["cues"].get(k), bool):
                    bad(f"labels.cues.{k} not bool")
        if (r.get("reliability") or {}).get("resolved_by") not in RESOLVED_BY:
            bad("bad reliability.resolved_by")
        ctx = r.get("context")
        if not isinstance(ctx, dict):
            bad("context missing")
        else:
            if r.get("record_type") == "comment" and ctx.get("submission") is None:
                bad("comment row with null context.submission")
            if r.get("record_type") == "submission" and ctx.get("submission") is not None:
                bad("submission row must have null context.submission")
    return problems


DATA_DIR = locate_data(CFG.dataset_version)
_rows = _read_jsonl(DATA_DIR / f"dataset-{CFG.dataset_version}.jsonl")
_problems = validate_rows(_rows)
assert not _problems, f"{len(_problems)} contract violations; first: {_problems[:3]}"
df = pd.DataFrame(_rows)
# ISO8601: the exports mix second- and microsecond-precision timestamps
df["created_dt"] = pd.to_datetime(df["created_utc"], utc=True, format="ISO8601")

corpus = pd.DataFrame(_read_jsonl(DATA_DIR / f"corpus-{CFG.dataset_version}.jsonl"))
corpus["created_dt"] = pd.to_datetime(corpus["created_utc"], utc=True, format="ISO8601")
card = json.loads((DATA_DIR / "dataset_card.json").read_text(encoding="utf-8"))

sarcastic = df["labels"].map(lambda l: bool(l["sarcastic"]))
language = df["labels"].map(lambda l: l["language"])
strat_key = language + "|sarc=" + sarcastic.astype(str)
# §7.2: null sampling_strategy (pilot) counts as natural; only explicit
# keyword_oversampled rows are excluded from natural-distribution metrics
natural = df["sampling_strategy"].map(lambda s: s != "keyword_oversampled")

IDENTITY = {k: card.get(k) for k in ("dataset_version", "prompt_version", "uyam_commit")}
print(f"loaded {len(df)} rows, {int(sarcastic.sum())} sarcastic | corpus {len(corpus)} rows")
print(f"dataset identity: {IDENTITY}")

# what the export does NOT carry — printed every run so no downstream cell
# silently assumes a channel exists (docs/UYAM_HANDOFF.md)
HAS_CUES = df["labels"].map(lambda l: l.get("cues") is not None).any()
HAS_TX = df["aux"].map(lambda a: a.get("tx_sentiment") is not None).any()
CTX_SOURCES = set(df["context"].map(lambda c: c.get("source", "annotator_snapshot")))
print(f"uncollected fields  → cues={HAS_CUES}, aux.tx_sentiment={HAS_TX}, "
      f"human_gold={df['human_gold'].notna().any()}")
print(f"conversational context source: {CTX_SOURCES}")
if CTX_SOURCES != {"annotator_snapshot"}:
    print("  WARNING (§11.7): context is rebuilt from the corpus dump, NOT the "
          "snapshot the annotators saw. Report as a limitation until uyam "
          "exports the snapshot (handoff H2).")
_missing_sent = int(df["labels"].map(lambda l: l["intended_sentiment"] is None).sum())
print(f"rows with an unresolved intended_sentiment (excluded from RQ3): {_missing_sent}")

### §10 data-readiness gate — training becomes legitimate only when ALL pass

In [ ]:
@dataclass
class GateReport:
    checks: list = field(default_factory=list)

    @property
    def passed(self) -> bool:
        return all(ok for _, ok, _ in self.checks)

    def render(self) -> str:
        lines = ["§10 data-readiness gate:"]
        lines += [f"  [{'PASS' if ok else 'FAIL'}] {n} — {d}" for n, ok, d in self.checks]
        lines.append("  => GATE PASSED: real training is legitimate." if self.passed else
                     "  => GATE FAILED: SMOKE mode only; no reported metrics.")
        return "\n".join(lines)


def _vnum(v) -> int:
    try:
        return int(str(v).lstrip("sarc-v").lstrip("v") or 0)
    except ValueError:
        return 0


def readiness_gate() -> GateReport:
    g = GateReport()
    dv, pv = card.get("dataset_version"), card.get("prompt_version")
    g.checks.append(("dataset-v2 under sarc-v2", _vnum(dv) >= 2 and _vnum(pv) >= 2,
                     f"dataset_version={dv}, prompt_version={pv}"))
    n_pos = int(sarcastic.sum())
    g.checks.append(("≥400 sarcastic positives", n_pos >= 400, f"{n_pos} positives"))
    gold = (card.get("agreement") or {}).get("gold_vs_ensemble_cohen_kappa") or {}
    g.checks.append(("gold subset labeled + κ reported",
                     int(gold.get("n_gold_items") or 0) >= 250 and gold.get("sarcastic") is not None,
                     f"n_gold_items={gold.get('n_gold_items')}, sarcastic κ={gold.get('sarcastic')}"))
    thin = {k: int(v) for k, v in strat_key.value_counts().items() if v < 10}
    g.checks.append(("every language×sarcastic cell ≥10", not thin,
                     f"thin cells: {thin}" if thin else "all cells ≥10"))
    g.checks.append(("fold file frozen", (RESULTS / f"folds-{CFG.dataset_version}.json").exists(),
                     f"results/folds-{CFG.dataset_version}.json"))
    return g


GATE = readiness_gate()
if GATE.passed:
    print("gate passed — smoke mode may be disabled deliberately in configs")
else:
    print("=" * 72)
    print(" SMOKE MODE — every number in this notebook is a harness or")
    print(" architecture check on the pilot export, NOT a reportable result.")
    print("=" * 72)
print(GATE.render())
SMOKE = CFG.tag()

## 4 · EDA (rerun on every dataset version — MODEL_PLAN §5)

In [ ]:
cells = pd.crosstab(language, sarcastic)
print(f"{SMOKE}sarcastic: {sarcastic.sum()} / {len(df)} ({sarcastic.mean():.1%})")
print(f"\n{SMOKE}language × sarcastic cells:")
print(cells)
for l in cells.index:
    for s in cells.columns:
        if cells.loc[l, s] < 10:
            print(f"  FLAG: cell ({l}, sarcastic={s}) has {cells.loc[l, s]} rows (<10)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
cells.plot.bar(ax=axes[0], title=f"{SMOKE}language × sarcastic")
df["sampling_strategy"].fillna("null (→natural)").value_counts().plot.bar(
    ax=axes[1], title=f"{SMOKE}sampling_strategy")
df["reliability"].map(lambda r: r["resolved_by"]).value_counts().plot.bar(
    ax=axes[2], title=f"{SMOKE}resolved_by")
plt.tight_layout(); plt.show()

rel_votes = df["reliability"].map(lambda r: r["sarcasm_votes"])
print(f"{SMOKE}votes × resolution:")
print(pd.crosstab(rel_votes, df["reliability"].map(lambda r: r["resolved_by"])))

### Text lengths vs the stage-1 truncation budgets (192 / 96 / 128 tokens)

In [ ]:
_tok_eda = AutoTokenizer.from_pretrained(CFG.encoder_name)
target_tokens = df["text"].map(lambda t: len(_tok_eda(t, truncation=False)["input_ids"]))
print(f"{SMOKE}target tokens: median {target_tokens.median():.0f}, "
      f"over budget ({CFG.max_len_target}): {(target_tokens > CFG.max_len_target).mean():.1%}")
plt.figure(figsize=(7, 3))
plt.hist(target_tokens, bins=30)
plt.axvline(CFG.max_len_target, color="r", ls="--", label=f"budget {CFG.max_len_target}")
plt.title(f"{SMOKE}target token counts"); plt.legend(); plt.tight_layout(); plt.show()
print("note: text carries collection-time mojibake (e.g. â€™ for ’); labels were "
      "conditioned on the text AS-IS, so it is never 'fixed' (§11.7).")

### Context coverage — conversational and temporal (viability panels)

In [ ]:
conv_stats = pd.DataFrame([{
    "has_submission": r["context"]["submission"] is not None,
    "n_parents": len(r["context"]["parent_chain"] or []),
    "n_replies": len(r["context"]["replies"] or []),
} for _, r in df.iterrows()])
print(f"{SMOKE}% with submission snapshot: {conv_stats['has_submission'].mean():.1%} | "
      f"≥1 parent: {(conv_stats['n_parents'] > 0).mean():.1%} "
      f"(mean chain {conv_stats['n_parents'].mean():.2f}) | "
      f"≥1 reply: {(conv_stats['n_replies'] > 0).mean():.1%}")

_by_author = {a: g.sort_values("created_dt")[["created_dt", "reddit_fullname"]].values.tolist()
              for a, g in corpus.groupby("author_hash", sort=False)}
prior_counts = df.apply(lambda r: sum(
    1 for dt, fid in _by_author.get(r["author_hash"], [])
    if dt < r["created_dt"] and fid != r["reddit_fullname"]), axis=1)
plt.figure(figsize=(7, 3))
plt.hist(prior_counts, bins=range(0, int(prior_counts.max()) + 2))
plt.title(f"{SMOKE}prior posts per target author (temporal-context viability)")
plt.tight_layout(); plt.show()
for k in (1, 3, CFG.temporal_k):
    print(f"{SMOKE}rows with ≥{k} prior posts: {(prior_counts >= k).mean():.1%}")

### Agreement statistics (echoed from dataset_card.json — cite, don't recompute)

In [ ]:
print(pd.DataFrame({k: {"fleiss_kappa": v["fleiss_kappa"], "krippendorff_alpha": v["krippendorff_alpha"]}
                    for k, v in card["agreement"]["labels"].items()}).T)
gold = card["agreement"]["gold_vs_ensemble_cohen_kappa"]
print(f"\ngold subset n={gold['n_gold_items']} → human-vs-ensemble κ not yet available")

### Manual read — every sarcastic row with all three annotator rationales

In [ ]:
for _, r in df[sarcastic].iterrows():
    lab = r["labels"]
    print("=" * 100)
    print(f"[{r['reddit_fullname']}] lang={lab['language']} literal={lab['literal_sentiment']} "
          f"intended={lab['intended_sentiment']} votes={r['reliability']['sarcasm_votes']}")
    print(f"TEXT: {r['text'][:300]}")
    for a in r["reliability"]["annotators"]:
        # per-annotator confidence is not exported; reliability.mean_confidence
        # is the row-level number (handoff H12)
        print(f"  {a['model_key']:<8} sarcastic={a['sarcastic']} "
              f"{a['literal_sentiment']}→{a['intended_sentiment']} | {a['rationale']}")

## 5 · Frozen folds (§7.1)

`StratifiedGroupKFold(5)` — stratify `sarcastic×language`, **group by
`submission_fullname`** (thread-mates share context; splitting a thread across
folds is leakage — §11.2). Val is carved from the train side with the same
grouping (≈80/10/10). Frozen once per dataset version; identity-checked on load.

In [ ]:
def make_folds(n_splits: int = 5, seed: int = 13, smoke: bool = True) -> list[dict]:
    y, groups, ids = strat_key.to_numpy(), df["submission_fullname"].to_numpy(), df["reddit_fullname"].to_numpy()
    with warnings.catch_warnings():
        if smoke:
            warnings.filterwarnings("ignore", message="The least populated class")
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = []
        for fold_i, (train_idx, test_idx) in enumerate(sgkf.split(df, y, groups)):
            gss = GroupShuffleSplit(n_splits=1, test_size=1 / 9, random_state=seed + fold_i)
            tr_rel, val_rel = next(gss.split(train_idx, groups=groups[train_idx]))
            folds.append({"fold": fold_i,
                          "train": ids[train_idx[tr_rel]].tolist(),
                          "val": ids[train_idx[val_rel]].tolist(),
                          "test": ids[test_idx].tolist()})
    thread_of = dict(zip(df["reddit_fullname"], df["submission_fullname"]))
    pos_ids = set(df.loc[sarcastic, "reddit_fullname"])
    for fold in folds:
        parts = {p: {thread_of[i] for i in fold[p]} for p in ("train", "val", "test")}
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            assert not parts[a] & parts[b], f"fold {fold['fold']}: thread spans {a}/{b}"
        for p in ("train", "val", "test"):
            if not pos_ids & set(fold[p]):
                msg = f"fold {fold['fold']} {p} has 0 sarcastic positives"
                if smoke:
                    print(f"WARNING (tolerated on pilot): {msg}")
                else:
                    raise ValueError(msg + " — not valid for real training")
    return folds


FOLDS_FILE = RESULTS / f"folds-{CFG.dataset_version}.json"
if FOLDS_FILE.exists():
    payload = json.loads(FOLDS_FILE.read_text(encoding="utf-8"))
    assert payload["identity"] == IDENTITY, (  # §11.8: never mix dataset versions
        f"frozen folds were built for {payload['identity']}, current export is "
        f"{IDENTITY} — regenerate folds for this version")
    FOLDS = payload["folds"]
    print(f"loaded frozen folds (identity verified): {FOLDS_FILE}")
else:
    FOLDS = make_folds(CFG.n_folds, seed=13, smoke=CFG.smoke)
    FOLDS_FILE.write_text(json.dumps(
        {"identity": IDENTITY, "n_splits": CFG.n_folds, "seed": 13, "folds": FOLDS},
        indent=2), encoding="utf-8")
    print(f"froze {len(FOLDS)} folds → {FOLDS_FILE}")

print(f"\n{SMOKE}fold composition (rows / sarcastic positives):")
pos_ids = set(df.loc[sarcastic, "reddit_fullname"])
print(pd.DataFrame([{"fold": f["fold"], **{p: f"{len(f[p])}/{len(pos_ids & set(f[p]))}"
                                           for p in ("train", "val", "test")}} for f in FOLDS]
                   ).to_string(index=False))

author_of = dict(zip(df["reddit_fullname"], df["author_hash"]))
test_authors = [{author_of[i] for i in f["test"]} for f in FOLDS]
multi = sum(1 for a in set().union(*test_authors) if sum(a in s for s in test_authors) > 1)
print(f"\n{SMOKE}author-overlap audit: {multi} authors appear in >1 test fold "
      "(if a few authors dominate later exports, add author_hash to the grouping key)")

## 6 · Context channels (§4.2–§4.3, §6)

- **conversational** — ONLY the embedded snapshot the annotators saw (§11.7)
- **temporal** — corpus posts by the same author strictly BEFORE the target
- **retrieval** — per-fold banks from TRAINING rows only, same-thread excluded
  (§11.1), leakage asserted at build time

In [ ]:
ROLE_SUBMISSION, ROLE_ANCESTOR, ROLE_REPLY = 0, 1, 2
ROLE_NAMES = {0: "submission", 1: "ancestor", 2: "reply"}


@dataclass
class ConvItem:
    text: str
    role: int
    is_submitter: bool


def build_conversational(row) -> list[ConvItem]:
    ctx = row["context"]
    items = []
    sub = ctx.get("submission")
    if sub is not None:
        text = "\n\n".join(t for t in (sub.get("title"), sub.get("selftext")) if t)
        if text.strip():
            items.append(ConvItem(text, ROLE_SUBMISSION, True))
    for p in ctx.get("parent_chain") or []:
        if (p.get("text") or "").strip():
            items.append(ConvItem(p["text"], ROLE_ANCESTOR, bool(p.get("is_submitter"))))
    for rep in ctx.get("replies") or []:
        if (rep.get("text") or "").strip():
            items.append(ConvItem(rep["text"], ROLE_REPLY, bool(rep.get("is_submitter"))))
    return items


@dataclass
class TemporalItem:
    text: str
    delta_days: float
    reddit_fullname: str


class TemporalIndex:
    def __init__(self, corpus_df: pd.DataFrame):
        self._by_author = {
            a: list(zip(g["created_dt"], g["reddit_fullname"], g["text"]))
            for a, g in corpus_df[["author_hash", "created_dt", "reddit_fullname", "text"]]
            .sort_values("created_dt").groupby("author_hash", sort=False)}

    def history(self, author_hash, target_dt, target_fullname, k=10) -> list[TemporalItem]:
        prior = [(dt, fid, tx) for dt, fid, tx in self._by_author.get(author_hash, [])
                 if dt < target_dt and fid != target_fullname and (tx or "").strip()]
        return [TemporalItem(tx, (target_dt - dt).total_seconds() / 86400.0, fid)
                for dt, fid, tx in reversed(prior[-k:])]  # most recent first


class RetrievalEmbeddings:
    """Frozen sentence embeddings for all targets, cached under cache/."""

    def __init__(self, fullnames, vectors):
        self.fullnames = list(fullnames)
        self.vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12)
        self.index = {f: i for i, f in enumerate(self.fullnames)}

    @classmethod
    def build(cls, cfg: Config) -> "RetrievalEmbeddings":
        cache = CACHE / f"retrieval-{cfg.dataset_version}-{cfg.retrieval_model.replace('/', '__')}.npz"
        fullnames = df["reddit_fullname"].tolist()
        if cache.exists():
            z = np.load(cache, allow_pickle=True)
            if z["fullnames"].tolist() == fullnames:
                return cls(fullnames, z["vectors"])
        from sentence_transformers import SentenceTransformer
        vectors = SentenceTransformer(cfg.retrieval_model).encode(
            df["text"].tolist(), batch_size=64, show_progress_bar=False, convert_to_numpy=True)
        np.savez_compressed(cache, fullnames=np.array(fullnames, dtype=object), vectors=vectors)
        return cls(fullnames, vectors)


@dataclass
class RetrievalResult:
    sarc: list
    nonsarc: list


def assert_no_leakage(retrieval: dict, train_set: set) -> None:
    """§11.1 hard checks: bank ⊆ training fold, never the query's own thread."""
    thread_of = dict(zip(df["reddit_fullname"], df["submission_fullname"]))
    for q, res in retrieval.items():
        for f in res.sarc + res.nonsarc:
            assert f in train_set, f"retrieval leakage: {f} outside the training fold (query {q})"
            assert thread_of[f] != thread_of[q], f"retrieval leakage: {f} shares thread with {q}"
            assert f != q, f"retrieval leakage: {q} retrieved itself"


def build_fold_retrieval(emb: RetrievalEmbeddings, train_fullnames, query_fullnames,
                         k=5) -> dict:
    thread_of = dict(zip(df["reddit_fullname"], df["submission_fullname"]))
    label_of = dict(zip(df["reddit_fullname"], sarcastic))
    banks = {True: [], False: []}
    for f in train_fullnames:
        banks[label_of[f]].append(f)
    out = {}
    for q in query_fullnames:
        q_vec, q_thread = emb.vectors[emb.index[q]], thread_of[q]
        picked = {}
        for lab, bank in banks.items():
            cands = [f for f in bank if f != q and thread_of[f] != q_thread]
            if not cands:
                picked[lab] = []
                continue
            sims = emb.vectors[[emb.index[f] for f in cands]] @ q_vec
            picked[lab] = [cands[i] for i in np.argsort(-sims)[:k]]
        out[q] = RetrievalResult(sarc=picked[True], nonsarc=picked[False])
    assert_no_leakage(out, set(train_fullnames))
    return out


# build the fold-independent channels once
CONV = {r["reddit_fullname"]: build_conversational(r) for _, r in df.iterrows()}
TINDEX = TemporalIndex(corpus)
TEMP = {r["reddit_fullname"]: TINDEX.history(r["author_hash"], r["created_dt"],
                                             r["reddit_fullname"], k=CFG.temporal_k)
        for _, r in df.iterrows()}
EMB = RetrievalEmbeddings.build(CFG)
_deltas = [it.delta_days for items in TEMP.values() for it in items]
assert all(d > 0 for d in _deltas), "temporal strictly-before violated"
print(f"{SMOKE}conversational items: {sum(len(v) for v in CONV.values())} | "
      f"temporal items: {len(_deltas)} | retrieval embeddings: {EMB.vectors.shape}")

### Audit — exactly which texts entered each channel (5 random rows, fold-0 banks)

In [ ]:
_fold0 = FOLDS[0]
_ret_demo = build_fold_retrieval(EMB, _fold0["train"],
                                 _fold0["train"] + _fold0["val"] + _fold0["test"],
                                 k=CFG.retrieval_k)
print(f"leakage assertions PASSED for {len(_ret_demo)} queries "
      f"(banks ⊆ {len(_fold0['train'])} train rows, same-thread excluded)")

_rows_by_name = {r["reddit_fullname"]: r for _, r in df.iterrows()}
_text_of = dict(zip(df["reddit_fullname"], df["text"]))
_clip = lambda t: (" ".join(t.split()))[:110] + ("…" if len(t) > 110 else "")
rng = np.random.default_rng(13)
for f in rng.choice(list(_ret_demo.keys()), size=5, replace=False):
    r = _rows_by_name[f]
    print(f"=== {f} ({r['labels']['language']}, sarcastic={r['labels']['sarcastic']}) ===")
    print(f"TARGET: {_clip(r['text'])}")
    for i, it in enumerate(CONV[f]):
        print(f"  conv {i}. {ROLE_NAMES[it.role]:<10} is_submitter={it.is_submitter} | {_clip(it.text)}")
    for i, it in enumerate(TEMP[f]):
        print(f"  temp {i}. Δt={it.delta_days:8.2f} d | {_clip(it.text)}")
    for x in _ret_demo[f].sarc:
        print(f"  ret S: {_clip(_text_of[x])}")
    for x in _ret_demo[f].nonsarc:
        print(f"  ret N: {_clip(_text_of[x])}")
    print()

## 7 · Model — the 5-stage architecture (§4), all behind ablation flags

One class: `use_conv=use_temp=use_ret=False` reduces it EXACTLY to the RQ1
context-agnostic baseline (`MLP([t])`), so the comparison can never drift.
Disabled channels are removed from the gate softmax (renormalized), not
zero-filled (§7.3).

In [ ]:
CHANNELS = ("conv", "temp", "ret")


def scatter_items(flat, batch_idx, batch_size):
    """[N, d] flattened items + owner index → padded [B, M, d] + bool mask [B, M]."""
    d = flat.shape[-1]
    counts = torch.bincount(batch_idx, minlength=batch_size)
    max_items = int(counts.max().item()) if counts.numel() and counts.max() > 0 else 1
    out = flat.new_zeros(batch_size, max_items, d)
    mask = torch.zeros(batch_size, max_items, dtype=torch.bool, device=flat.device)
    slot = torch.zeros(batch_size, dtype=torch.long, device=flat.device)
    for n in range(flat.shape[0]):
        b = batch_idx[n]
        out[b, slot[b]] = flat[n]
        mask[b, slot[b]] = True
        slot[b] += 1
    return out, mask


class SharedEncoder(nn.Module):
    """Stage 1: ONE XLM-R for every text unit + pooling + projection (§4.1)."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(cfg.encoder_name)
        self.pooling = cfg.pooling
        self.proj = nn.Linear(self.backbone.config.hidden_size, cfg.d_model)
        if cfg.freeze_bottom_layers > 0:
            for p in self.backbone.embeddings.parameters():
                p.requires_grad = False
            for layer in self.backbone.encoder.layer[:cfg.freeze_bottom_layers]:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
        else:
            pooled = h[:, 0]
        return self.proj(pooled)

    def forward_chunked(self, input_ids, attention_mask, chunk=64):
        if input_ids.shape[0] <= chunk:
            return self.forward(input_ids, attention_mask)
        return torch.cat([self.forward(input_ids[i:i + chunk], attention_mask[i:i + chunk])
                          for i in range(0, input_ids.shape[0], chunk)])


class Tokenize:
    def __init__(self, cfg: Config):
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_name)
        self.budgets = {"target": cfg.max_len_target, "context": cfg.max_len_context,
                        "selftext": cfg.max_len_selftext}

    def __call__(self, texts, kind="context"):
        if not texts:
            return {"input_ids": torch.zeros(0, 1, dtype=torch.long),
                    "attention_mask": torch.zeros(0, 1, dtype=torch.long)}
        enc = self.tokenizer(texts, truncation=True, max_length=self.budgets[kind],
                             padding=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}


class TargetAttention(nn.Module):
    """Stage-2/3 block: scaled dot-product attention, target as query."""

    def __init__(self, d):
        super().__init__()
        self.w_q, self.w_k, self.w_v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.scale = math.sqrt(d)

    def forward(self, target, items, mask):
        q = self.w_q(target).unsqueeze(1)
        k, v = self.w_k(items), self.w_v(items)
        scores = (q @ k.transpose(1, 2)).squeeze(1) / self.scale
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = torch.where((~mask.any(-1)).unsqueeze(-1), torch.zeros_like(attn), attn)
        return (attn.unsqueeze(1) @ v).squeeze(1)  # empty rows → zeros


class ContextAwareSarcasmModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.encoder = SharedEncoder(cfg)
        self.active = [c for c, on in zip(CHANNELS, (cfg.use_conv, cfg.use_temp, cfg.use_ret)) if on]

        if cfg.use_conv:
            self.conv_attn = TargetAttention(d)
            if cfg.conv_role_embeddings:
                self.role_emb = nn.Embedding(3, d)
                self.submitter_emb = nn.Embedding(2, d)
        if cfg.use_temp:
            self.temp_attn = TargetAttention(d)
            raw = math.log(math.expm1(cfg.temporal_lambda_init))  # softplus(raw)=λ0
            self.temporal_lambda_raw = nn.Parameter(torch.tensor(raw, dtype=torch.float32),
                                                    requires_grad=cfg.temporal_lambda_learnable)
        if cfg.use_ret:
            self.ret_attn_sarc = TargetAttention(d)
            self.ret_attn_nonsarc = TargetAttention(d)
            self.ret_proj = nn.Linear(2 * d, d)
        if self.active and cfg.missing_channel == "learned":
            self.missing_emb = nn.ParameterDict({c: nn.Parameter(torch.zeros(d)) for c in self.active})
        if len(self.active) >= 2:  # stage 4: gate over ACTIVE channels only
            self.gate = nn.Linear(len(self.active) * d, len(self.active))
        in_dim = d * (2 if self.active else 1)
        self.classifier = nn.Sequential(nn.Linear(in_dim, cfg.mlp_hidden), nn.GELU(),
                                        nn.Dropout(cfg.dropout), nn.Linear(cfg.mlp_hidden, 2))
        if cfg.aux_cue_heads:
            self.cue_head = nn.Linear(in_dim, 4)
        if cfg.aux_polarity_shift:
            self.shift_head = nn.Linear(in_dim, 2)
        if cfg.aux_language_head:
            self.lang_head = nn.Linear(in_dim, 3)
        # RQ3 stage-2 head is trained POST-HOC on frozen features (§13), never here

    @property
    def temporal_lambda(self):
        return F.softplus(self.temporal_lambda_raw)

    def _encode_items(self, part, batch_size):
        if part["input_ids"].shape[0] == 0:
            dev = next(self.parameters()).device
            return (torch.zeros(batch_size, 1, self.cfg.d_model, device=dev),
                    torch.zeros(batch_size, 1, dtype=torch.bool, device=dev))
        flat = self.encoder.forward_chunked(part["input_ids"], part["attention_mask"])
        return scatter_items(flat, part["batch_idx"], batch_size)

    def _apply_missing(self, out, empty, channel):
        """empty: [B] bool — rows whose channel had no items at all."""
        if self.cfg.missing_channel == "learned":
            fill = self.missing_emb[channel].to(out.dtype)
            out = torch.where(empty.unsqueeze(-1), fill.expand_as(out), out)
        return out  # "zeros": attention already yields zeros for empty rows

    def _conv_channel(self, batch, t):
        items, mask = self._encode_items(batch["conv"], t.shape[0])
        if self.cfg.conv_role_embeddings and batch["conv"]["input_ids"].shape[0] > 0:
            extra_flat = self.role_emb(batch["conv"]["role"]) + self.submitter_emb(batch["conv"]["is_submitter"])
            extra, _ = scatter_items(extra_flat, batch["conv"]["batch_idx"], t.shape[0])
            items = items + extra
        return self._apply_missing(self.conv_attn(t, items, mask), ~mask.any(-1), "conv")

    def _temp_channel(self, batch, t):
        items, mask = self._encode_items(batch["temp"], t.shape[0])
        if batch["temp"]["input_ids"].shape[0] > 0:
            decay_flat = torch.exp(-self.temporal_lambda * batch["temp"]["delta_days"])
            decay, _ = scatter_items(decay_flat.unsqueeze(-1), batch["temp"]["batch_idx"], t.shape[0])
            items = items * decay.to(items.dtype)  # K/V · exp(−λ·Δt), §4.2
        return self._apply_missing(self.temp_attn(t, items, mask), ~mask.any(-1), "temp")

    def _ret_channel(self, batch, t):
        s_items, s_mask = self._encode_items(batch["ret_sarc"], t.shape[0])
        n_items, n_mask = self._encode_items(batch["ret_nonsarc"], t.shape[0])
        out = self.ret_proj(torch.cat([self.ret_attn_sarc(t, s_items, s_mask),
                                       self.ret_attn_nonsarc(t, n_items, n_mask)], dim=-1))
        # banks pad to different widths (sarcastic bank can hold <k exemplars):
        # combine emptiness per row, never mask | mask
        return self._apply_missing(out, ~(s_mask.any(-1) | n_mask.any(-1)), "ret")

    def forward(self, batch):
        t = self.encoder(batch["target"]["input_ids"], batch["target"]["attention_mask"])
        out = {"target_emb": t}
        chans = []
        for name in self.active:
            c = {"conv": self._conv_channel, "temp": self._temp_channel,
                 "ret": self._ret_channel}[name](batch, t)
            chans.append(c)
            out[f"c_{name}"] = c
        if not self.active:
            feats, out["gates"] = t, None
        elif len(self.active) == 1:
            feats = torch.cat([t, chans[0]], dim=-1)
            out["gates"] = torch.ones(t.shape[0], 1, device=t.device)
        else:
            g = F.softmax(self.gate(torch.cat(chans, dim=-1)), dim=-1)
            feats = torch.cat([t, sum(g[:, i:i + 1] * chans[i] for i in range(len(chans)))], dim=-1)
            out["gates"] = g
        out["features"] = feats
        out["logits"] = self.classifier(feats)
        if self.cfg.aux_cue_heads:
            out["cue_logits"] = self.cue_head(feats)
        if self.cfg.aux_polarity_shift:
            out["shift_logits"] = self.shift_head(feats)
        if self.cfg.aux_language_head:
            out["lang_logits"] = self.lang_head(feats)
        return out


print("model classes defined")

## 8 · Training harness — loss (§4.5 + §9 flags), fold loop, gate enforcement

In [ ]:
LANG2ID = {"english": 0, "tagalog": 1, "taglish": 2}


class SarcasmDataset(Dataset):
    def __init__(self, fullnames, retrieval):
        self.samples = [_rows_by_name[f] for f in fullnames]
        self.retrieval = retrieval or {}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        r = self.samples[i]
        f, lab, rel = r["reddit_fullname"], r["labels"], r["reliability"]
        ret = self.retrieval.get(f)
        p_conf = 0.95 if (rel.get("sarcasm_votes") or "") == "3-0" else 0.75  # §9.1
        return {
            "fullname": f, "target_text": r["text"],
            "conv": CONV.get(f, []), "temp": TEMP.get(f, []),
            "ret_sarc": [_text_of[x] for x in ret.sarc] if ret else [],
            "ret_nonsarc": [_text_of[x] for x in ret.nonsarc] if ret else [],
            "label": int(lab["sarcastic"]),
            "soft_pos": p_conf if lab["sarcastic"] else 1.0 - p_conf,
            # uyam collects neither cue labels nor a guaranteed sentiment
            # pair; -1 masks the aux losses for rows that cannot supervise them
            "cues": ([float(lab["cues"][k]) for k in CUE_KEYS]
                     if lab.get("cues") else [-1.0] * len(CUE_KEYS)),
            "shift": (int(lab["literal_sentiment"] != lab["intended_sentiment"])
                      if lab["literal_sentiment"] and lab["intended_sentiment"] else -1),
            "lang": LANG2ID[lab["language"]],
            "resolved_by": rel["resolved_by"],
        }


class Collator:
    def __init__(self, cfg: Config, tok: Tokenize):
        self.cfg, self.tok = cfg, tok

    def _flatten(self, per_sample, extras=None):
        texts, batch_idx, flat_extras = [], [], []
        for b, items in enumerate(per_sample):
            for j, t in enumerate(items):
                texts.append(t)
                batch_idx.append(b)
                if extras is not None:
                    flat_extras.append(extras[b][j])
        out = {**self.tok(texts), "batch_idx": torch.tensor(batch_idx, dtype=torch.long)}
        if extras is not None:
            out["extras"] = flat_extras
        return out

    def __call__(self, samples):
        cfg = self.cfg
        batch = {
            "fullnames": [s["fullname"] for s in samples],
            "target": self.tok([s["target_text"] for s in samples], kind="target"),
            "labels": torch.tensor([s["label"] for s in samples], dtype=torch.long),
            "soft_pos": torch.tensor([s["soft_pos"] for s in samples]),
            "cues": torch.tensor([s["cues"] for s in samples]),
            "shift": torch.tensor([s["shift"] for s in samples], dtype=torch.long),
            "lang": torch.tensor([s["lang"] for s in samples], dtype=torch.long),
            "sample_weight": torch.tensor(
                [cfg.sample_weights.get(s["resolved_by"], 1.0) if cfg.sample_weighting else 1.0
                 for s in samples]),
        }
        if cfg.use_conv:
            conv = self._flatten([[it.text for it in s["conv"]] for s in samples],
                                 [[(it.role, it.is_submitter) for it in s["conv"]] for s in samples])
            ex = conv.pop("extras")
            conv["role"] = torch.tensor([e[0] for e in ex], dtype=torch.long)
            conv["is_submitter"] = torch.tensor([int(e[1]) for e in ex], dtype=torch.long)
            batch["conv"] = conv
        if cfg.use_temp:
            temp = self._flatten([[it.text for it in s["temp"]] for s in samples],
                                 [[it.delta_days for it in s["temp"]] for s in samples])
            temp["delta_days"] = torch.tensor(temp.pop("extras"))
            batch["temp"] = temp
        if cfg.use_ret:
            batch["ret_sarc"] = self._flatten([s["ret_sarc"] for s in samples])
            batch["ret_nonsarc"] = self._flatten([s["ret_nonsarc"] for s in samples])
        return batch


def to_device(batch, device):
    return {k: ({kk: (vv.to(device) if torch.is_tensor(vv) else vv) for kk, vv in v.items()}
                if isinstance(v, dict) else (v.to(device) if torch.is_tensor(v) else v))
            for k, v in batch.items()}


def class_weights(train_fullnames):
    """Inverse-frequency (neg, pos) on ONE training fold — never global."""
    y = np.array([int(_rows_by_name[f]["labels"]["sarcastic"]) for f in train_fullnames])
    n, n_pos = len(y), int(y.sum())
    if n_pos == 0 or n_pos == n:
        print(f"WARNING: degenerate training fold (n_pos={n_pos}) — equal weights")
        return 1.0, 1.0
    return n / (2.0 * (n - n_pos)), n / (2.0 * n_pos)


def compute_loss(out, batch, cfg, class_w):
    logits, labels = out["logits"], batch["labels"]
    if cfg.soft_labels:
        soft = torch.stack([1.0 - batch["soft_pos"], batch["soft_pos"]], dim=-1)
        per = -(soft * F.log_softmax(logits, dim=-1)).sum(-1) * class_w[labels]
    elif cfg.focal_loss:
        ce = F.cross_entropy(logits, labels, weight=class_w, reduction="none")
        per = ((1 - torch.exp(-ce)) ** cfg.focal_gamma) * ce
    else:
        per = F.cross_entropy(logits, labels, weight=class_w, reduction="none")
    loss = (per * batch["sample_weight"]).mean()
    aux = 0.0
    if cfg.aux_cue_heads:
        cue_mask = batch["cues"] >= 0  # -1 = label not collected
        if cue_mask.any():
            aux = aux + F.binary_cross_entropy_with_logits(
                out["cue_logits"][cue_mask], batch["cues"][cue_mask])
    if cfg.aux_polarity_shift:
        # ignore_index skips rows whose sentiment pair never resolved
        aux = aux + F.cross_entropy(out["shift_logits"], batch["shift"], ignore_index=-1)
    if cfg.aux_language_head:
        aux = aux + F.cross_entropy(out["lang_logits"], batch["lang"])
    return loss + cfg.aux_loss_weight * aux if isinstance(aux, torch.Tensor) else loss


def build_optimizer(model, cfg):
    enc_ids = {id(p) for p in model.encoder.backbone.parameters()}
    enc = [p for p in model.parameters() if id(p) in enc_ids and p.requires_grad]
    heads = [p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
    if cfg.layerwise_lr_decay is None:
        groups = [{"params": enc, "lr": cfg.lr_encoder}, {"params": heads, "lr": cfg.lr_heads}]
    else:  # §9.4
        layers = model.encoder.backbone.encoder.layer
        n = len(layers)
        groups = [{"params": heads, "lr": cfg.lr_heads},
                  {"params": [p for p in model.encoder.backbone.embeddings.parameters() if p.requires_grad],
                   "lr": cfg.lr_encoder * cfg.layerwise_lr_decay ** n}]
        groups += [{"params": [p for p in l.parameters() if p.requires_grad],
                    "lr": cfg.lr_encoder * cfg.layerwise_lr_decay ** (n - 1 - i)}
                   for i, l in enumerate(layers)]
    return torch.optim.AdamW(groups)


def build_scheduler(optimizer, total_steps, warmup_ratio):
    warmup = max(1, int(total_steps * warmup_ratio))
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: s / warmup if s < warmup
        else max(0.0, (total_steps - s) / max(1, total_steps - warmup)))


def safe_f1(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(f1_score(y_true, y_pred, zero_division=0))


def predict(model, loader, cfg, collect_features=False):
    model.eval()
    rows, feats, targets = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                out = model(batch)
            probs = F.softmax(out["logits"].float(), -1)[:, 1].cpu().numpy()
            gates = out["gates"].float().cpu().numpy() if out["gates"] is not None else None
            if collect_features:
                feats.append(out["features"].float().cpu().numpy())
                targets.append(out["target_emb"].float().cpu().numpy())
            for i, f in enumerate(batch["fullnames"]):
                row = {"reddit_fullname": f, "y_true": int(batch["labels"][i]),
                       "prob": float(probs[i]), "pred": int(probs[i] >= 0.5)}
                if gates is not None:
                    for j, name in enumerate(model.active):
                        row[f"gate_{name}"] = float(gates[i, j])
                rows.append(row)
    pred = pd.DataFrame(rows)
    if collect_features:
        pred.attrs["features"] = np.concatenate(feats) if feats else np.zeros((0,))
        # target-only embeddings: the RQ3 stage-1 head is context-free by
        # definition (thesis §3.5 — "applied directly to the original post")
        pred.attrs["target_emb"] = np.concatenate(targets) if targets else np.zeros((0,))
    return pred


def train_model(model, train_loader, val_loader, cfg, class_w=(1.0, 1.0), log_every=0):
    """Early-stopped training on val F1; restores the best weights."""
    model.to(DEVICE)
    optimizer = build_optimizer(model, cfg)
    steps = len(train_loader) if cfg.max_steps_per_epoch is None else min(
        len(train_loader), cfg.max_steps_per_epoch)
    scheduler = build_scheduler(optimizer, max(1, steps * cfg.max_epochs // cfg.grad_accum),
                                cfg.warmup_ratio)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)
    w = torch.tensor(class_w, dtype=torch.float, device=DEVICE)
    best_f1, best_state, patience_left = -1.0, None, cfg.patience
    hist = {"train_loss": [], "val_f1": []}
    for epoch in range(cfg.max_epochs):
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            if cfg.max_steps_per_epoch is not None and step >= cfg.max_steps_per_epoch:
                break
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                loss = compute_loss(model(batch), batch, cfg, w) / cfg.grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            losses.append(float(loss.item()) * cfg.grad_accum)
        val = predict(model, val_loader, cfg)
        val_f1 = safe_f1(val["y_true"].to_numpy(), val["pred"].to_numpy())
        hist["train_loss"].append(float(np.mean(losses)) if losses else float("nan"))
        hist["val_f1"].append(val_f1)
        if log_every:
            print(f"{cfg.tag()}epoch {epoch}: train_loss={hist['train_loss'][-1]:.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, patience_left = val_f1, cfg.patience
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    hist["best_val_f1"] = best_f1
    return hist


def make_loaders(cfg, fold, tok, seed):
    retrieval = build_fold_retrieval(EMB, fold["train"],
                                     fold["train"] + fold["val"] + fold["test"],
                                     k=cfg.retrieval_k) if cfg.use_ret else None
    collate = Collator(cfg, tok)
    loaders = {}
    for part in ("train", "val", "test"):
        ds = SarcasmDataset(fold[part], retrieval)
        if part == "train":
            if cfg.weighted_sampler:  # §9.2
                y = np.array([int(r["labels"]["sarcastic"]) for r in ds.samples])
                pos_frac = max(y.mean(), 1e-6)
                w = np.where(y == 1, cfg.target_pos_frac / pos_frac,
                             (1 - cfg.target_pos_frac) / max(1 - pos_frac, 1e-6))
                loaders[part] = DataLoader(ds, batch_size=cfg.batch_size, collate_fn=collate,
                                           sampler=WeightedRandomSampler(
                                               torch.tensor(w, dtype=torch.double), len(ds), True))
            else:
                loaders[part] = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                                           generator=torch.Generator().manual_seed(seed),
                                           collate_fn=collate)
        else:
            loaders[part] = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)
    return loaders


def safe_metrics(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {"f1": float(f1_score(y_true, y_pred, zero_division=0)),
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "accuracy": float(accuracy_score(y_true, y_pred)) if len(y_true) else float("nan"),
                "n": int(len(y_true)), "n_pos": int(np.sum(y_true))}


META = pd.DataFrame({"reddit_fullname": df["reddit_fullname"], "language": language,
                     "record_type": df["record_type"], "natural": natural,
                     "resolved_by": df["reliability"].map(lambda r: r["resolved_by"])})


def run_cv(cfg: Config, verbose=True):
    """Folds × seeds loop; writes results/<run_name>/. HARD RULE: refuses
    non-smoke runs while the §10 gate fails."""
    if not GATE.passed and not cfg.smoke:
        raise RuntimeError("§10 readiness gate FAILED — real training refused:\n" + GATE.render())
    tok = Tokenize(cfg)
    fold_rows, pred_frames = [], []
    for fold in FOLDS:
        for seed in cfg.seeds:
            set_seed(seed)
            loaders = make_loaders(cfg, fold, tok, seed)
            w = class_weights(fold["train"]) if cfg.class_weighting == "inverse_freq" else (1.0, 1.0)
            model = ContextAwareSarcasmModel(cfg)
            hist = train_model(model, loaders["train"], loaders["val"], cfg, w)
            pred = predict(model, loaders["test"], cfg)
            pred["fold"], pred["seed"] = fold["fold"], seed
            pred = pred.merge(META, on="reddit_fullname", how="left")
            pred_frames.append(pred)
            m = safe_metrics(pred["y_true"].to_numpy(), pred["pred"].to_numpy())
            fold_rows.append({"fold": fold["fold"], "seed": seed, **m,
                              "best_val_f1": hist["best_val_f1"]})
            if verbose:
                print(f"{cfg.tag()}fold {fold['fold']} seed {seed}: test_f1={m['f1']:.4f} "
                      f"(n_pos={m['n_pos']}) val_f1={hist['best_val_f1']:.4f}")
            del model
            torch.cuda.empty_cache()
    fold_metrics, predictions = pd.DataFrame(fold_rows), pd.concat(pred_frames, ignore_index=True)
    out_dir = RESULTS / cfg.run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = "SMOKE-" if cfg.smoke else ""
    fold_metrics.to_csv(out_dir / f"{prefix}fold_metrics.csv", index=False)
    predictions.to_csv(out_dir / f"{prefix}predictions.csv", index=False)
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                text=True, check=True).stdout.strip()
    except Exception:
        commit = "unknown"
    (out_dir / f"{prefix}run.json").write_text(json.dumps(
        {"config": cfg.to_dict(), "dataset_identity": IDENTITY, "leische_commit": commit,
         "gate_passed": GATE.passed}, indent=2), encoding="utf-8")
    return {"fold_metrics": fold_metrics, "predictions": predictions}


def fold_summary(fm):
    """mean ± std across folds × seeds (§11.6: never report single-run F1)."""
    out = {c: f"{fm[c].mean():.4f} ± {0.0 if np.isnan(fm[c].std()) else fm[c].std():.4f}"
           for c in ("f1", "precision", "recall", "accuracy")}
    return pd.DataFrame([{"runs": len(fm), **out}])


def natural_and_all(pred):
    """§7.2 hygiene: natural-only vs all-rows metrics."""
    return pd.DataFrame([
        {"slice": "natural_only", **safe_metrics(pred[pred["natural"]]["y_true"].to_numpy(),
                                                 pred[pred["natural"]]["pred"].to_numpy())},
        {"slice": "all_rows", **safe_metrics(pred["y_true"].to_numpy(), pred["pred"].to_numpy())}])


def slice_metrics(pred, by):
    return pd.DataFrame([{by: v, **safe_metrics(s["y_true"].to_numpy(), s["pred"].to_numpy())}
                         for v, s in pred.groupby(by)])


def ece(y_true, prob, n_bins=10):
    if len(prob) == 0:
        return float("nan")
    bins, err = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if m.sum():
            err += (m.sum() / len(prob)) * abs(prob[m].mean() - y_true[m].mean())
    return float(err)


print("training harness ready")

## 9 · Smoke test 1 — overfit 16 rows to ~zero loss (§10)

All 8 pilot positives + 8 negatives. A healthy architecture must memorize 16
rows; failure means wiring bugs, not data problems. Run for the baseline
(target-only) and the full model (conv+temp+ret).

In [ ]:
def overfit_smoke(cfg: Config, n=16, max_steps=150, target_loss=0.05, seed=13):
    cfg = copy.deepcopy(cfg)
    cfg.smoke = True  # overfitting is never a real result
    set_seed(seed)
    chosen = (df.loc[sarcastic, "reddit_fullname"].tolist()
              + df.loc[~sarcastic, "reddit_fullname"].tolist())[:n]
    retrieval = build_fold_retrieval(EMB, chosen, chosen, k=cfg.retrieval_k) if cfg.use_ret else None
    loader = DataLoader(SarcasmDataset(chosen, retrieval), batch_size=cfg.batch_size,
                        shuffle=True, generator=torch.Generator().manual_seed(seed),
                        collate_fn=Collator(cfg, Tokenize(cfg)))
    model = ContextAwareSarcasmModel(cfg).to(DEVICE)
    opt_cfg = copy.deepcopy(cfg)
    opt_cfg.lr_heads = 1e-3  # hotter head lr for memorization
    optimizer = build_optimizer(model, opt_cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)
    w = torch.tensor([1.0, 1.0], device=DEVICE)
    losses, step = [], 0
    model.train()
    while step < max_steps:
        for batch in loader:
            if step >= max_steps:
                break
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                loss = compute_loss(model(batch), batch, cfg, w)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.item()))
            step += 1
            if len(losses) >= 5 and np.mean(losses[-5:]) < target_loss:
                step = max_steps
                break
    final = float(np.mean(losses[-5:]))
    print(f"{cfg.tag()}overfit-{len(chosen)}: final mean loss {final:.4f} "
          f"({len(losses)} steps) → {'PASS' if final < target_loss else 'FAIL'}")
    del model
    torch.cuda.empty_cache()
    return {"losses": losses, "passed": final < target_loss}


ov_base = overfit_smoke(smoke_cfg())
ov_full = overfit_smoke(smoke_cfg(use_conv=True, use_temp=True, use_ret=True))
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, ov, title in ((axes[0], ov_base, "baseline (target-only)"),
                      (axes[1], ov_full, "full model (conv+temp+ret)")):
    ax.plot(ov["losses"])
    ax.axhline(0.05, color="r", ls="--")
    ax.set_title(f"{SMOKE}overfit-16 — {title}")
plt.tight_layout(); plt.show()
assert ov_base["passed"] and ov_full["passed"], "architecture failed to memorize 16 rows"

## 10 · Smoke test 2 — tiny-settings 5-fold dry run (baseline) + pooling check

In [ ]:
for pooling in ("mean", "cls"):  # §4.1 asks to verify both — one fold each at SMOKE scale
    cfg_p = smoke_cfg(pooling=pooling, run_name=f"smoke-pooling-{pooling}")
    set_seed(13)
    tok_p = Tokenize(cfg_p)
    loaders = make_loaders(cfg_p, FOLDS[0], tok_p, 13)
    model = ContextAwareSarcasmModel(cfg_p)
    hist = train_model(model, loaders["train"], loaders["val"], cfg_p, class_weights(FOLDS[0]["train"]))
    pred = predict(model, loaders["test"], cfg_p)
    m = safe_metrics(pred["y_true"].to_numpy(), pred["pred"].to_numpy())
    print(f"{SMOKE}pooling={pooling}: fold-0 test F1={m['f1']:.4f} (n_pos={m['n_pos']})")
    del model
    torch.cuda.empty_cache()
print(f"{SMOKE}both pooling paths run; thesis default stays pooling=mean.\n")

res_base = run_cv(smoke_cfg(run_name="smoke-baseline"))
print(f"\n{SMOKE}5-fold dry run, mean ± std across folds (1 seed, tiny settings):")
print(fold_summary(res_base["fold_metrics"]).to_string(index=False))
print(f"\n{SMOKE}§7.2 hygiene split (pilot has no keyword_oversampled rows — path exercised):")
print(natural_and_all(res_base["predictions"]).to_string(index=False))
print(f"\n{SMOKE}F1 by language (empty-positive folds are the pilot's stratification failure):")
print(slice_metrics(res_base["predictions"], "language").to_string(index=False))
print(f"\n{SMOKE}confusion (rows=true, cols=pred):")
print(confusion_matrix(res_base["predictions"]["y_true"], res_base["predictions"]["pred"], labels=[0, 1]))

## 11 · Full context model — gates logged, checkpoint round-trip

Condition 8 on fold 0 at SMOKE settings. Per-instance gate values are logged
from day one (§4.4) — their distribution by language / record_type is
thesis-discussion material. Features `[t ; c_fused]` are kept for §13.

In [ ]:
cfg_full = smoke_cfg(use_conv=True, use_temp=True, use_ret=True, run_name="smoke-full")
set_seed(13)
tok_full = Tokenize(cfg_full)
loaders = make_loaders(cfg_full, FOLDS[0], tok_full, 13)
w0 = class_weights(FOLDS[0]["train"])
print(f"{SMOKE}fold-0 class weights (neg, pos): ({w0[0]:.3f}, {w0[1]:.3f})")
model_full = ContextAwareSarcasmModel(cfg_full)
hist = train_model(model_full, loaders["train"], loaders["val"], cfg_full, w0, log_every=1)
print(f"{SMOKE}best val F1: {hist['best_val_f1']:.4f} | temporal λ: "
      f"{float(model_full.temporal_lambda):.4f} (fixed — thesis baseline)")

pred_f0 = predict(model_full, loaders["test"], cfg_full).merge(META, on="reddit_fullname")
gate_cols = [c for c in pred_f0.columns if c.startswith("gate_")]
print(f"\n{SMOKE}gate distribution over fold-0 test rows:")
print(pred_f0[gate_cols].describe().loc[["mean", "std", "min", "max"]])
print(f"\n{SMOKE}mean gates by language:")
print(pred_f0.groupby("language")[gate_cols].mean())
fig, ax = plt.subplots(figsize=(7, 3))
for c in gate_cols:
    ax.hist(pred_f0[c], bins=15, alpha=0.6, label=c)
ax.set_title(f"{SMOKE}per-instance gate values (fold-0 test)"); ax.legend()
plt.tight_layout(); plt.show()

ckpt = CACHE / "smoke_full_ckpt.pt"
torch.save(model_full.state_dict(), ckpt)
model_check = ContextAwareSarcasmModel(cfg_full)
model_check.load_state_dict(torch.load(ckpt, weights_only=True))
model_check.to(DEVICE)
pred_check = predict(model_check, loaders["test"], cfg_full)
assert np.allclose(pred_f0["prob"].to_numpy(), pred_check["prob"].to_numpy(), atol=1e-6)
print(f"checkpoint round-trip OK → {ckpt}")
del model_check
torch.cuda.empty_cache()

# frozen features + sarcasm flags for ALL rows (fold-0 banks) — used by §13
all_names = df["reddit_fullname"].tolist()
_ret_all = build_fold_retrieval(EMB, FOLDS[0]["train"], all_names, k=cfg_full.retrieval_k)
loader_all = DataLoader(SarcasmDataset(all_names, _ret_all), batch_size=cfg_full.batch_size,
                        shuffle=False, collate_fn=Collator(cfg_full, tok_full))
pred_all = predict(model_full, loader_all, cfg_full, collect_features=True)
FEATURES = pred_all.attrs["features"]
TARGET_EMB = pred_all.attrs["target_emb"]
FLAGS = dict(zip(pred_all["reddit_fullname"], pred_all["pred"].astype(bool)))
PROBS = dict(zip(pred_all["reddit_fullname"], pred_all["prob"]))
print(f"{SMOKE}features {FEATURES.shape} + flags for all rows "
      f"({int(pred_all['pred'].sum())} flagged sarcastic — smoke-quality)")
del model_full
torch.cuda.empty_cache()

## 12 · Ablation matrix — 8 conditions × 5 folds (RQ2, §7.3) + significance

Ultra-tiny settings prove the loop and artifacts end-to-end. With ~1–2 test
positives per fold the metrics are noise by construction (§11.6).

In [ ]:
CONDITIONS = [("1_baseline", 0, 0, 0), ("2_conv", 1, 0, 0), ("3_temp", 0, 1, 0),
              ("4_ret", 0, 0, 1), ("5_conv_temp", 1, 1, 0), ("6_conv_ret", 1, 0, 1),
              ("7_temp_ret", 0, 1, 1), ("8_full", 1, 1, 1)]
ablation = {}
for name, c, t_, r_ in CONDITIONS:
    t0 = time.time()
    ablation[name] = run_cv(ablation_cfg(use_conv=bool(c), use_temp=bool(t_), use_ret=bool(r_),
                                         run_name=f"ablation-smoke-{name}"), verbose=False)
    fm = ablation[name]["fold_metrics"]
    print(f"{SMOKE}{name:<12} conv={c} temp={t_} ret={r_} | mean F1 {fm['f1'].mean():.3f} "
          f"| {time.time() - t0:5.1f}s")

table = pd.DataFrame([{
    "condition": name, "runs": len(res["fold_metrics"]),
    **{c: f"{res['fold_metrics'][c].mean():.4f} ± {0.0 if np.isnan(res['fold_metrics'][c].std()) else res['fold_metrics'][c].std():.4f}"
       for c in ("f1", "precision", "recall", "accuracy")}}
    for name, res in ablation.items()])
print(f"\n{SMOKE}RQ2 ablation matrix — HARNESS CHECK ONLY:")
print(table.to_string(index=False))
table.to_csv(RESULTS / "SMOKE-ablation-matrix.csv", index=False)

### Significance: condition 8 vs condition 1 (bootstrap, randomization, McNemar)

In [ ]:
def _aligned(a, b):
    return a[["reddit_fullname", "seed", "y_true", "pred"]].merge(
        b[["reddit_fullname", "seed", "pred"]], on=["reddit_fullname", "seed"],
        suffixes=("_a", "_b"))


def paired_bootstrap(a, b, n_boot=1000, seed=13):
    m = _aligned(a, b)
    rng = np.random.default_rng(seed)
    y, pa, pb = m["y_true"].to_numpy(), m["pred_a"].to_numpy(), m["pred_b"].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        diffs = np.array([f1_score(y[i], pb[i], zero_division=0) - f1_score(y[i], pa[i], zero_division=0)
                          for i in (rng.integers(0, len(y), len(y)) for _ in range(n_boot))])
        obs = f1_score(y, pb, zero_division=0) - f1_score(y, pa, zero_division=0)
    return {"observed": float(obs), "ci95": (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))),
            "p": float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())))}


def approx_randomization(a, b, n_iter=1000, seed=13):
    m = _aligned(a, b)
    rng = np.random.default_rng(seed)
    y, pa, pb = m["y_true"].to_numpy(), m["pred_a"].to_numpy(), m["pred_b"].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        obs = abs(f1_score(y, pb, zero_division=0) - f1_score(y, pa, zero_division=0))
        count = sum(1 for _ in range(n_iter)
                    if abs(f1_score(y, np.where(s := rng.random(len(y)) < 0.5, pa, pb), zero_division=0)
                           - f1_score(y, np.where(s, pb, pa), zero_division=0)) >= obs - 1e-12)
    return {"observed": float(obs), "p": float((count + 1) / (n_iter + 1))}


def mcnemar_test(a, b):
    from statsmodels.stats.contingency_tables import mcnemar
    m = _aligned(a, b)
    ok_a, ok_b = (m["pred_a"] == m["y_true"]).to_numpy(), (m["pred_b"] == m["y_true"]).to_numpy()
    tab = [[int((ok_a & ok_b).sum()), int((ok_a & ~ok_b).sum())],
           [int((~ok_a & ok_b).sum()), int((~ok_a & ~ok_b).sum())]]
    res = mcnemar(np.array(tab), exact=True)
    return {"table": tab, "p": float(res.pvalue)}


pa, pb = ablation["1_baseline"]["predictions"], ablation["8_full"]["predictions"]
boot = paired_bootstrap(pa, pb)
print(f"{SMOKE}paired bootstrap ΔF1 (full − baseline): {boot['observed']:+.4f} "
      f"CI95 [{boot['ci95'][0]:+.4f}, {boot['ci95'][1]:+.4f}] p={boot['p']:.3f}")
ar = approx_randomization(pa, pb)
print(f"{SMOKE}approximate randomization |ΔF1|={ar['observed']:.4f} p={ar['p']:.3f}")
mc = mcnemar_test(pa, pb)
print(f"{SMOKE}McNemar discordants {mc['table'][0][1]} vs {mc['table'][1][0]}: p={mc['p']:.3f}")
print(f"{SMOKE}with ~8 positives these p-values are definitionally meaningless — "
      "the deliverable is that the machinery runs.")
print(f"\n{SMOKE}condition-8 F1 by resolved_by (§7.4):")
print(slice_metrics(pb, "resolved_by").to_string(index=False))

## 13 · RQ3 — two-stage sentiment evaluation (§8, thesis §3.5)

Ground truth `labels.intended_sentiment`; rows whose sentiment vote never
resolved are excluded (they carry no ground truth).

- **Stage 1 (pre-sarcasm)** — thesis §3.5: a sentiment head trained on the
  annotated `literal_sentiment` and applied *directly to the target text*,
  i.e. over the **target-only** embedding, no context. (MODEL_PLAN §8 proposed
  the shipped `aux.tx_sentiment` instead; uyam does not collect it, so the
  manuscript's own formulation is what runs — handoff H3.)
- **Stage 2 (post-sarcasm)** — rows the sarcasm model flags are re-predicted by
  a head over the frozen `[t ; c_fused]` features trained on
  `intended_sentiment`. Unflagged rows keep their stage-1 prediction.

Both heads are fit on fold-0 **training** rows only, so RQ3 supervision never
touches the sarcasm encoder or the test fold. The interesting cell:
*sarcastic ∧ literal≠intended*.

In [ ]:
literal = df["labels"].map(lambda l: l["literal_sentiment"])
intended = df["labels"].map(lambda l: l["intended_sentiment"])
has_sent = literal.notna() & intended.notna()
shift = has_sent & (literal != intended)
print(f"{SMOKE}rows with a resolved sentiment pair: {int(has_sent.sum())} / {len(df)}")
print(f"{SMOKE}literal≠intended: {int(shift.sum())} | sarcastic ∧ shift: "
      f"{int((sarcastic & shift).sum())} rows")


def sentiment_metrics(y_true, y_pred):
    if len(y_true) == 0:  # small slices can be empty — NaN, not a crash
        return {"macro_f1": float("nan"), "accuracy": float("nan"), "n": 0}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {"macro_f1": float(f1_score(y_true, y_pred, labels=list(SENTIMENTS),
                                           average="macro", zero_division=0)),
                "accuracy": float(accuracy_score(y_true, y_pred)), "n": int(len(y_true))}


def rq3_report(sub: pd.DataFrame, pred: pd.Series, tag: str) -> pd.DataFrame:
    y = sub["labels"].map(lambda l: l["intended_sentiment"])
    lg = sub["labels"].map(lambda l: l["language"])
    sc = sub["labels"].map(lambda l: l["sarcastic"])
    lit = sub["labels"].map(lambda l: l["literal_sentiment"])
    rows = [{"slice": "overall", **sentiment_metrics(y, pred)}]
    rows += [{"slice": f"language={v}", **sentiment_metrics(y[lg == v], pred[lg == v])}
             for v in sorted(lg.unique())]
    rows += [{"slice": f"gold_sarcastic={s}", **sentiment_metrics(y[sc == s], pred[sc == s])}
             for s in (False, True)]
    m = sc & (lit != y)
    rows.append({"slice": "sarcastic ∧ literal≠intended", **sentiment_metrics(y[m], pred[m])})
    out = pd.DataFrame(rows)
    out.insert(0, "run", tag)
    return out


# RQ3 population: fold-0 rows that actually carry sentiment ground truth
feat_of = {f: i for i, f in enumerate(pred_all["reddit_fullname"])}
rq3_train = [f for f in FOLDS[0]["train"]
             if has_sent[df["reddit_fullname"] == f].iloc[0]]
test_df = df[df["reddit_fullname"].isin(FOLDS[0]["test"]) & has_sent]
print(f"\n{SMOKE}RQ3 fold-0: {len(rq3_train)} train rows, {len(test_df)} test rows "
      f"(dropped rows without a resolved sentiment pair)")

# stage 1 — thesis §3.5: trained on literal_sentiment, applied to the TARGET
# TEXT ALONE, so it is fit over the context-free target embedding
stage1_head = LogisticRegression(max_iter=2000)
stage1_head.fit(TARGET_EMB[[feat_of[f] for f in rq3_train]],
                [literal[df["reddit_fullname"] == f].iloc[0] for f in rq3_train])
s1 = pd.Series(stage1_head.predict(TARGET_EMB[[feat_of[f] for f in test_df["reddit_fullname"]]]),
               index=test_df.index)

# stage 2 — trained on intended_sentiment over the frozen [t ; c_fused] features
stage2_head = LogisticRegression(max_iter=2000)
stage2_head.fit(FEATURES[[feat_of[f] for f in rq3_train]],
                [intended[df["reddit_fullname"] == f].iloc[0] for f in rq3_train])
s2 = pd.Series(stage2_head.predict(FEATURES[[feat_of[f] for f in test_df["reddit_fullname"]]]),
               index=test_df.index)

flagged = pd.Series([FLAGS[f] for f in test_df["reddit_fullname"]], index=test_df.index)
final_pred = s2.where(flagged, s1)
print(f"{SMOKE}{int(flagged.sum())} of {len(test_df)} test rows flagged sarcastic "
      "→ re-interpreted by the stage-2 head\n")

print(f"{SMOKE}RQ3 comparison on fold-0 test rows (ground truth = intended_sentiment):")
print(pd.concat([rq3_report(test_df, s1, SMOKE + "stage1-only"),
                 rq3_report(test_df, final_pred, SMOKE + "two-stage")],
                ignore_index=True).to_string(index=False))

y_sarc = test_df["labels"].map(lambda l: int(l["sarcastic"])).to_numpy()
probs_f0 = np.array([PROBS[f] for f in test_df["reddit_fullname"]])
print(f"\n{SMOKE}sarcasm-flag ECE on fold-0 test: {ece(y_sarc, probs_f0):.4f} "
      "(§9.8: temperature_scaling / tune_threshold_on_val flags exist — post-gate)")

## 14 · Verdict & next steps

The pipeline is one config change away from real training. When **dataset-v2**
lands in uyam:
1. put the new export in `./data/` (or just have uyam cloned next to this repo),
2. set `Config.dataset_version = "v2"` and re-run top-to-bottom
   (folds re-freeze automatically for the new identity),
3. once the §10 gate prints PASS, run with `Config(smoke=False, ...)` —
   until then `run_cv` refuses non-smoke runs by design.

In [ ]:
print(f"{SMOKE}smoke test 1 (overfit-16, baseline + full): PASS")
print(f"{SMOKE}smoke test 2 (5-fold dry run): PASS")
print(f"{SMOKE}8×5 ablation matrix + significance tests: RAN END-TO-END")
print(f"{SMOKE}RQ3 two-stage pipeline: RAN END-TO-END")
print()
print(GATE.render())